# kural-astro: fine-tune a small LLM to triage classical Tamil verses for physical/cosmological content

Runtime: **GPU (T4 is enough)**. Upload `sft_train.jsonl` and `sft_val.jsonl` from `data/train/`.
Labels are *silver* (assigned by Claude, unverified), so every metric here is provisional.
The model learns to output JSON `{label, meaning, science_concept, caveat}` with labels
L literal / O observation-in-simile / A mystic-cosmological (analogy only) / M metaphor / N none.
It does NOT fetch real-world research; that is a separate retrieval layer.

In [ ]:
!pip install -q unsloth scikit-learn

In [ ]:
from google.colab import files
up = files.upload()  # choose sft_train.jsonl and sft_val.jsonl
print(list(up))

In [ ]:
from unsloth import FastLanguageModel
import torch, json

# 7B reads Tamil better than 3B but is much slower on a free T4; Drive checkpointing below is what protects you now. Swap freely: "unsloth/gemma-2-2b-it", "unsloth/Llama-3.2-3B-Instruct", ... Tamil ability of
# small models is weak; if val scores are poor on Thiruvarutpa (Tamil only), try a bigger/Indic model.
MODEL = "unsloth/Qwen2.5-7B-Instruct"
model, tok = FastLanguageModel.from_pretrained(MODEL, max_seq_length=1536, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0, bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", random_state=7)

In [ ]:
from datasets import load_dataset
ds = load_dataset("json", data_files={"train":"sft_train.jsonl","val":"sft_val.jsonl"})
def fmt(ex):
    return {"text": tok.apply_chat_template(ex["messages"], tokenize=False)}
train = ds["train"].map(fmt)
print(train[0]["text"][:600])

In [ ]:
# Save checkpoints to Drive so a disconnect does NOT mean starting over.
# First run: click the auth link. If Colab disconnects later, just re-run every cell from the
# top ONE more time (reinstall + reload model is unavoidable) -- but training itself will pick
# up from the last saved checkpoint instead of restarting at step 0.
from google.colab import drive
drive.mount('/content/drive')
import os
CKPT_DIR = "/content/drive/MyDrive/kural_astro_ckpt"
os.makedirs(CKPT_DIR, exist_ok=True)
RESUME = any(f.startswith("checkpoint") for f in os.listdir(CKPT_DIR))
print("resuming from checkpoint" if RESUME else "starting fresh", "->", CKPT_DIR)

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model=model, tokenizer=tok, train_dataset=train,
    args=SFTConfig(dataset_text_field="text", max_seq_length=1536,
        per_device_train_batch_size=2, gradient_accumulation_steps=8,
        num_train_epochs=2, learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.05,
        logging_steps=5, save_strategy="steps", save_steps=20, save_total_limit=2, optim="adamw_8bit", seed=7, output_dir=CKPT_DIR, report_to="none"))
from unsloth.chat_templates import train_on_responses_only
# learn only from the assistant answer (label + concept), not from the long prompt
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user"+chr(10), response_part="<|im_start|>assistant"+chr(10))
trainer.train(resume_from_checkpoint=RESUME)

In [ ]:
# Evaluate on validation (held-out chapters/sections)
from sklearn.metrics import classification_report, f1_score
FastLanguageModel.for_inference(model)
def predict(messages):
    prompt = tok.apply_chat_template(messages[:2], tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**ids, max_new_tokens=160, do_sample=False)
    txt = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
    try: return json.loads(txt)["label"]
    except Exception: return "BAD"
gold, pred = [], []
for ex in ds["val"]:
    gold.append(ex["label"]); pred.append(predict(ex["messages"]))
print("unparseable:", pred.count("BAD"), "of", len(pred))
print(classification_report(gold, pred, zero_division=0))
phys = lambda l: l in ("L","O","A")
print("binary physical-vs-not F1:", f1_score([phys(g) for g in gold],[phys(p) for p in pred]))
maj = max(set(gold), key=gold.count)
print("majority-class baseline accuracy:", gold.count(maj)/len(gold))

In [ ]:
# Try it on your own verse
def analyse(src, tamil, english="", commentary=""):
    u = f"Source: {src}\nTamil: {tamil}\n"
    if english: u += f"English: {english}\n"
    if commentary: u += f"Commentary: {commentary}\n"
    u += "Return JSON with keys label (L/O/A/M/N), meaning, science_concept, caveat."
    m = [{"role":"system","content":ds["train"][0]["messages"][0]["content"]},{"role":"user","content":u}]
    prompt = tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**ids, max_new_tokens=200, do_sample=False)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
print(analyse("test", "அண்டங்கள் எல்லாம் அணுவில் அடக்கும் அரும்பெருஞ் சித்தரே வாரீர்"))

In [ ]:
# SWEEP: run the trained model over verses it has never seen (upload sweep_inputs.jsonl)
up2 = files.upload()
rows = [json.loads(l) for l in open("sweep_inputs.jsonl", encoding="utf-8")]
res = []
for i, r in enumerate(rows):
    prompt = tok.apply_chat_template(r["messages"], tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**ids, max_new_tokens=200, do_sample=False)
    txt = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
    try: p = json.loads(txt)
    except Exception: p = {"label": "BAD", "raw": txt}
    res.append({"source": r["source"], "url": r["url"], **p})
    if i % 25 == 0: print(i, "/", len(rows))
with open("predictions.jsonl", "w", encoding="utf-8") as f:
    for x in res: f.write(json.dumps(x, ensure_ascii=False) + "\n")
files.download("predictions.jsonl")
from collections import Counter; print(Counter(x["label"] for x in res))

In [ ]:
model.save_pretrained("kural_astro_lora"); tok.save_pretrained("kural_astro_lora")
!zip -qr kural_astro_lora.zip kural_astro_lora
files.download("kural_astro_lora.zip")